# Module 5: Report Generation
**CAP6412 — Bias & Safety Auditor for T2I Models**

Auto-generates a structured PDF audit report with bias charts, safety statistics,
concept erasure comparison, methodology, and references.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import json
import pandas as pd

from src.report_generator import ReportGenerator

In [ ]:
# Check required results exist
for f in ['../results/bias_results.csv', '../results/safety_results.csv', '../results/bias_summary.json']:
    exists = Path(f).exists()
    print(f'  {"✓" if exists else "✗"} {f}')

In [ ]:
# Quick summary stats before generating report
bias_df   = pd.read_csv('../results/bias_results.csv')
safety_df = pd.read_csv('../results/safety_results.csv')
with open('../results/bias_summary.json') as f:
    summary = json.load(f)

print(f'Total images analysed: {len(bias_df)}')
print(f'NSFW flagged:          {safety_df["is_flagged"].sum()} / {len(safety_df)}')
print()
print('Top biased prompts by gender:')
top = (
    bias_df[bias_df['category'] == 'occupation']
    .groupby('prompt')['gender_pred']
    .apply(lambda x: (x == 'man').mean())
    .sort_values(ascending=False)
    .head(5)
)
for prompt, pct in top.items():
    print(f'  {prompt[:55]:<55} {pct:.0%} male')

In [ ]:
# Generate the PDF report
Path('../results').mkdir(exist_ok=True)
Path('../results/charts').mkdir(parents=True, exist_ok=True)

rg = ReportGenerator(
    bias_csv='../results/bias_results.csv',
    safety_csv='../results/safety_results.csv',
    bias_summary_json='../results/bias_summary.json',
    output_pdf='../results/audit_report.pdf',
    charts_dir='../results/charts',
    erasure_dir='../results/erasure_comparison',  # optional
)

report_path = rg.build_pdf()
print(f'Report: {report_path}')

In [ ]:
# List generated charts
charts = list(Path('../results/charts').glob('*.png'))
print(f'Generated {len(charts)} charts:')
for c in charts:
    print(f'  {c.name}')

In [ ]:
# Preview charts inline
import matplotlib.pyplot as plt
from PIL import Image

for chart_path in sorted(Path('../results/charts').glob('*.png'))[:4]:
    img = Image.open(chart_path)
    plt.figure(figsize=(8, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(chart_path.stem, fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
print('All deliverables:')
deliverables = [
    '../prompts.csv',
    '../results/bias_results.csv',
    '../results/safety_results.csv',
    '../results/bias_summary.json',
    '../results/audit_report.pdf',
    '../models/erased_unet',
]
for d in deliverables:
    exists = Path(d).exists()
    print(f'  {"✓" if exists else "✗"} {d}')